# Comparing Trivy scanning modes — filesystem, image, and repository

Trivy supports three primary scanning modes: `trivy fs` (filesystem), `trivy image` (container image), and `trivy repo` (remote repository). Each targets a different attack surface. This notebook runs all three against the same sample project and compares the output structure, depth, and practical use cases.

## Purpose

- Understand the structural and functional differences between the three modes
- Build a decision framework for choosing the right mode in CI/CD and ad-hoc workflows
- Identify which vulnerabilities each mode catches and which it misses

The three modes share the same vulnerability database and severity engine, but differ in:
- **Scan target** — what they point at (filesystem path, image reference, Git URL)
- **Scope** — what they can discover (OS packages, language deps, misconfigurations, secrets)
- **Output fidelity** — how they report findings (per-file vs per-package vs per-layer)

## When to use each mode

| Mode | Best for | Limitations |
|------|----------|-------------|
| `trivy fs` | Local code scan during development, pre-commit, pre-PR | No registry auth needed; can't detect base-image vulns without pulling |
| `trivy image` | Container image vulnerability scanning; registry-integrated workflows | Requires image pull; no misconfig scan for infra-as-code |
| `trivy repo` | Git repository scanning on push / schedule; no local clone needed | Slower (clone + scan); no secrets scan in `repo` mode directly |

## Prerequisites

- `trivy` installed (v0.50+ recommended: `brew install trivy` or `apt install trivy`)
- Python 3.8+ with `jq` for JSON processing
- Docker daemon running (for `trivy image` mode)
- Network access to GitHub and Docker Hub

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

def check_trivy():
    result = subprocess.run(["trivy", "--version"], capture_output=True, text=True)
    if result.returncode != 0:
        print("trivy not found. Install: https://trivy.dev/latest/docs/getting-started/installation/")
        sys.exit(1)
    print(f"Using: {result.stdout.splitlines()[0]}")

check_trivy()

## Step 1: Prepare a sample project

Create a small Python project with known-vulnerable dependencies to scan. This project includes a `requirements.txt` with outdated packages and a Dockerfile.

In [ ]:
SAMPLE_DIR = Path(tempfile.mkdtemp(prefix="trivy-compare-"))
print(f"Sample project: {SAMPLE_DIR}")

# requirements.txt with intentionally old dependencies
(SAMPLE_DIR / "requirements.txt").write_text("""\
requests==2.25.1
flask==2.0.1
pyyaml==5.3.1
""")

# Dockerfile using an old base image
(SAMPLE_DIR / "Dockerfile").write_text("""\
FROM python:3.9-slim-buster
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
""")

# Minimal app.py
(SAMPLE_DIR / "app.py").write_text("""\
from flask import Flask
import requests

app = Flask(__name__)

@app.route("/")
def index():
    return requests.get("https://example.com").text
""")

print("Sample project created with:")
for f in SAMPLE_DIR.iterdir():
    print(f"  {f.name}")

## Step 2: Run `trivy fs` — filesystem scan

Filesystem mode scans a local directory for OS packages, language dependencies, misconfigurations, and secrets. It does not need a container runtime and works entirely offline after the DB download.

In [ ]:
FS_OUT = SAMPLE_DIR / "trivy-fs-result.json"

cmd = [
    "trivy", "fs",
    "--format", "json",
    "--output", str(FS_OUT),
    "--quiet",
    "--severity", "CRITICAL,HIGH,MEDIUM",
    str(SAMPLE_DIR)
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode not in (0, 1):
    print(f"trivy fs failed (exit {result.returncode}): {result.stderr.strip()}")
    sys.exit(1)

with open(FS_OUT) as f:
    fs_data = json.load(f)
print(f"trivy fs complete. Results: {FS_OUT}")

## Step 3: Run `trivy image` — container image scan

Image mode scans a pulled container image. It detects OS-level vulnerabilities from the base image (e.g., Debian Buster package CVEs) plus language dependencies installed in the image layers.

In [ ]:
# Build the sample image
IMAGE_TAG = "trivy-compare-sample:latest"
build = subprocess.run(
    ["docker", "build", "-t", IMAGE_TAG, str(SAMPLE_DIR)],
    capture_output=True, text=True
)
if build.returncode != 0:
    print(f"Docker build failed: {build.stderr.strip()}")
    sys.exit(1)
print(f"Image built: {IMAGE_TAG}")

# Scan with trivy image
IMG_OUT = SAMPLE_DIR / "trivy-image-result.json"
cmd = [
    "trivy", "image",
    "--format", "json",
    "--output", str(IMG_OUT),
    "--quiet",
    "--severity", "CRITICAL,HIGH,MEDIUM",
    IMAGE_TAG
]
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode not in (0, 1):
    print(f"trivy image failed (exit {result.returncode}): {result.stderr.strip()}")
    sys.exit(1)

with open(IMG_OUT) as f:
    img_data = json.load(f)
print(f"trivy image complete. Results: {IMG_OUT}")

## Step 4: Run `trivy repo` — remote repository scan

Repository mode clones a Git repo and scans its contents. This is useful for CI workflows scanning incoming pull requests or scheduled repo audits.

In [ ]:
# Use a small public repo with known vulnerabilities for demonstration
REPO_URL = "https://github.com/velancio/vulnerable-flask-app"
REPO_OUT = SAMPLE_DIR / "trivy-repo-result.json"

cmd = [
    "trivy", "repo",
    "--format", "json",
    "--output", str(REPO_OUT),
    "--quiet",
    "--severity", "CRITICAL,HIGH,MEDIUM",
    REPO_URL
]
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
if result.returncode not in (0, 1):
    print(f"trivy repo failed (exit {result.returncode}): {result.stderr.strip()}")
    sys.exit(1)

with open(REPO_OUT) as f:
    repo_data = json.load(f)
print(f"trivy repo complete. Results: {REPO_OUT}")

## Step 5: Compare results

Extract the same metrics from each mode: total results, result types, and coverage.

In [ ]:
def summarize_results(data, label):
    results = data.get("Results", [])
    total = sum(len(r.get("Vulnerabilities", [])) for r in results)
    misconfigs = sum(len(r.get("Misconfigurations", [])) for r in results)
    secrets = sum(len(r.get("Secrets", [])) for r in results)
    targets = [r.get("Target", "unknown") for r in results]
    types = set(r.get("Type", "") for r in results)
    print(f"\n--- {label} ---")
    print(f"  Targets scanned: {len(results)}")
    for t in targets:
        print(f"    - {t}")
    print(f"  Result types: {', '.join(sorted(types)) or 'none'}")
    print(f"  Vulnerabilities: {total}")
    print(f"  Misconfigurations: {misconfigs}")
    print(f"  Secrets: {secrets}")
    return total, misconfigs, secrets

fs_total, fs_mis, fs_sec = summarize_results(fs_data, "trivy fs")
img_total, img_mis, img_sec = summarize_results(img_data, "trivy image")
repo_total, repo_mis, repo_sec = summarize_results(repo_data, "trivy repo")

## Step 6: Analysis — what each mode found

The key differences typically observed:

- `trivy fs` detects **language-level vulnerabilities** (pip, npm) and **misconfigurations** (Dockerfile, Kubernetes manifests, Terraform) but **does not** detect OS package CVEs unless the OS packages are present in the scanned filesystem (e.g., a chroot or container rootfs).

- `trivy image` detects **both OS-level and language-level vulnerabilities** because it inspects the full image filesystem including the base OS layer. It also reports misconfigurations in Dockerfiles embedded in the image history.

- `trivy repo` clones the repository and runs filesystem-mode scans on the checkout. It's functionally similar to `trivy fs` but operates on remote repositories. It supports the same scanners (vuln, misconfig, secret) but requires network access for cloning.

In [ ]:
# Build a comparison table
print(f"{'Metric':<35} {'trivy fs':<15} {'trivy image':<15} {'trivy repo':<15}")
print("-" * 80)
print(f"{'Vulnerabilities found':<35} {fs_total:<15} {img_total:<15} {repo_total:<15}")
print(f"{'Misconfigurations':<35} {fs_mis:<15} {img_mis:<15} {repo_mis:<15}")
print(f"{'Secrets':<35} {fs_sec:<15} {img_sec:<15} {repo_sec:<15}")
print()
print("Note: trivy image typically shows higher counts because it sees OS-level")
print("packages from the base image (e.g., glibc, openssl) that fs and repo modes miss.")

## Step 7: Decision guide for CI/CD

| CI scenario | Recommended mode | Rationale |
|---|---|---|
| Pre-commit hook | `trivy fs .` | Fast, no network needed after DB cached, catches code-level issues |
| PR check (container) | `trivy image` | Validates the built image including base image supply chain |
| PR check (IaC) | `trivy fs --scanners misconfig` | Focuses on Terraform/K8s misconfigs without noise from unrelated vulns |
| Scheduled scan | `trivy repo` + `trivy image` | Both: repo for dependency drift, image for running container base |
| Multi-stage build | `trivy image --ignore-unfixed` | Flags actionable vulns in final stage only |
| Monorepo service audit | `trivy fs` per service dir | Individual scan per service with per-service severity thresholds |

## Step 8: Common gotchas

- **`trivy fs` on a container rootfs** — If you mount a container's extracted filesystem, `trivy fs` can detect OS vulns, but it won't have layer metadata. Use `trivy image` instead for accurate layer attribution.
- **`trivy repo` auth** — Private repos require `--token` or `GITHUB_TOKEN` environment variable. Without auth, the clone fails silently.
- **Severity filtering mismatch** — `trivy image --severity CRITICAL,HIGH` filters at the scanner level, not post-scan. To re-filter output, use `jq` on JSON output.
- **SBOM re-scanning** — All three modes can output CycloneDX/SPDX SBOMs (`--format cyclonedx`), but only `trivy image` can scan an existing SBOM with `trivy sbom`.

## Verify

This notebook confirms:
1. `trivy fs` covers language deps, misconfigs, and secrets but **not** OS base-image vulns
2. `trivy image` covers OS + language vulns plus image-layer metadata but **not** secrets or misconfigs by default (`--scanners` flag can enable them)
3. `trivy repo` mirrors `trivy fs` behaviour against a remote Git URL

The choice between modes depends primarily on what stage of the pipeline you are in and what attack surface you need to cover. A robust CI pipeline runs two modes: `trivy repo` on push (for IaC and dependency misconfig) and `trivy image` on the built artifact (for container supply chain).

## References

- [Trivy documentation — scanning modes](https://trivy.dev/latest/docs/)
- [Trivy GitHub repository](https://github.com/aquasecurity/trivy)
- [Trivy SARIF output format](https://trivy.dev/latest/docs/references/output/sarif/)